# ◆ LUMEN Protocol — an agent society that *thinks* inside your Colab

**MVM (Rust) + PDB + three real MIT agents (Astrid · Iris · Elena) + a local LLM on your GPU** — free, no API keys, no cloud.

What this notebook does, step by step:

1. Clones [lumen-protocol](https://github.com/GonzaloMonzonC/lumen-protocol) and the three reference agents: [Astrid](https://github.com/GonzaloMonzonC/astrid) (evidence), [Iris](https://github.com/GonzaloMonzonC/iris) (hypotheses), [Elena](https://github.com/GonzaloMonzonC/elena) (decisions). All MIT.
2. Builds (or downloads) the Rust **MVM** as a shared library.
3. Creates a throwaway **PDB** and seeds it with demo state.
4. **GPU LLM (free):** installs `llama.cpp` (CUDA) and serves **Qwen2.5-14B** on the T4 (3B fallback on CPU) at `127.0.0.1:58099` — the agents think with a model that lives *inside this Colab*. No internet, no keys, no bill.
5. Runs the **A·I·E cycle** on a topic *you choose*: Iris proposes a falsifiable hypothesis → Astrid checks it against registered evidence → Elena signs a **decision card** (criterion + owner + review date). Real routines, real PDB, local LLM.
7. **Notary contract:** same state → same digest → same `cid` (sha256).
8. **Ask the system (interactive):** a chat UI — pick a sibling (or the council) and hold a real conversation (with memory). Every exchange is recorded in the PDB.
9. **Full arsenal:** the MVM speaks **MCP natively** (a local MCP server serving this very PDB — `mcp:list` / `mcp:call` from Rust, no bridge), **browses the web** via its HTTP device, and Elena signs a decision card with that live data in hand.
10. **Bonus:** local semantic search over the digest claims (fastembed).

> Runtime: **Runtime ▸ Change runtime type ▸ T4 GPU** recommended. Everything works on CPU too (the LLM will be slower).

*Astrid: "no source, no claim." Iris: "possibilities, never facts." Elena: "no criterion, no owner, no date — then it is not a decision."*


In [ ]:
# 1. Environment + clone (idempotent)
import os, sys, time, platform, subprocess, shutil
REPO    = "/content/lumen-protocol"
ASTRID  = "/content/astrid"
IRIS    = "/content/iris"
ELENA   = "/content/elena"
for path, url in [
    (REPO,   "https://github.com/GonzaloMonzonC/lumen-protocol.git"),
    (ASTRID, "https://github.com/GonzaloMonzonC/astrid.git"),
    (IRIS,   "https://github.com/GonzaloMonzonC/iris.git"),
    (ELENA,  "https://github.com/GonzaloMonzonC/elena.git"),
]:
    if not os.path.exists(path):
        subprocess.run(["git", "clone", "--depth", "1", url, path], check=True)
print("python:", platform.python_version())
print("cargo:", shutil.which("cargo") or "NOT INSTALLED (will install in step 2)")
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True, timeout=10)
    GPU = gpu.stdout.strip()
    print("GPU:", GPU or "none")
except Exception:
    GPU = ""
    print("GPU: none detected")


In [ ]:
# 2. Rust toolchain + MVM shared library (prebuilt .so or local build)
if not shutil.which("cargo"):
    print("Installing rustup...")
    subprocess.run(["bash", "-c",
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y"], check=True)
    os.environ["PATH"] = os.environ.get("PATH", "") + ":" + os.path.expanduser("~/.cargo/bin")
sys.path.insert(0, REPO + "/implementations/mcp-servers/pdb")
import lumen_mlight as lm
print("wrapper loaded; lib target:", lm._lib_path())
import urllib.request
try:
    _so_dir = os.path.dirname(lm._lib_path())
    os.makedirs(_so_dir, exist_ok=True)
    _so_path = os.path.join(_so_dir, os.path.basename(lm._lib_path()))
    if not os.path.exists(_so_path):
        _urls = [
            "https://github.com/GonzaloMonzonC/lumen-protocol/releases/latest/download/liblumen_mlight.so",
            "https://github.com/GonzaloMonzonC/lumen-protocol/releases/download/mlight-v0.1.0/liblumen_mlight.so",
        ]
        for _u in _urls:
            try:
                urllib.request.urlretrieve(_u, _so_path)
                print("prebuilt .so downloaded:", os.path.getsize(_so_path), "bytes")
                break
            except Exception as _e:
                print("fallback:", _e)
except Exception as e:
    print("note:", e)


In [ ]:
# 3. Throwaway PDB + demo state (real M code, real MVM)
DB = "/content/lumen_demo.db"
if os.path.exists(DB):
    os.remove(DB)

seed = """
S ^ACTIVE="astrid-demo"
S ^ANGI("metrics","agents_online")="{\\"value\\": 3, \\"updated\\": \\"2026-09-10T12:00:00Z\\"}"
S ^ANGI("metrics","alerts_last_run")="0"
S ^SYS("MCP","worker1","type")="http"
S ^SYS("MCP","worker1","url")="https://worker1.example/mcp"
S ^QUANTUM("colapso",1)="{\\"idx\\": 1, \\"backend\\": \\"sim\\"}"
W "seed ok"
"""
r = lm.execute(seed, routines={}, sqlite_path=DB, gas_limit=50000)
r2 = lm.execute('W "active=",$G(^ACTIVE)," workers=",$O(^SYS("MCP",""))', routines={}, sqlite_path=DB, gas_limit=5000)
out2 = (r2.get("state") or {}).get("output", "")
print("check:", out2)
assert "astrid-demo" in str(out2), "seed did not persist"
print("PDB ready:", DB)


In [ ]:
# 4. Local LLM — free forever: llama.cpp serving a real Qwen on 127.0.0.1:58099
#    The MVM's $DEVICE("llm:call", ..., "local", model) calls this endpoint directly.
#    No API keys. No cloud. The model lives inside this Colab.
#    First run: ~2-7 min (download + load), depending on the model selected below.
import subprocess, sys, time, os, json, urllib.request

# Auto-select: the T4's 15 GB comfortably fits the 14B (Q4_K_M ≈ 9 GB).
if GPU:
    MODEL_FILE = "/content/qwen2.5-14b-instruct-q4_k_m.gguf"
    MODEL_URL  = "https://huggingface.co/bartowski/Qwen2.5-14B-Instruct-GGUF/resolve/main/Qwen2.5-14B-Instruct-Q4_K_M.gguf"
    print("GPU detected → loading the big one: Qwen2.5-14B-Instruct Q4_K_M (~9 GB, fits the T4)")
else:
    MODEL_FILE = "/content/qwen2.5-3b-instruct-q4_k_m.gguf"
    MODEL_URL  = "https://huggingface.co/Qwen/Qwen2.5-3B-Instruct-GGUF/resolve/main/qwen2.5-3b-instruct-q4_k_m.gguf"
    print("No GPU runtime → 3B on CPU (works, but slow — T4 recommended)")

# 4a. llama-cpp-python with CUDA wheels (fast). Fallback: CPU wheel.
try:
    import llama_cpp  # noqa
    print("llama_cpp already installed")
except ImportError:
    print("installing llama-cpp-python (CUDA)...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python[server]",
                        "--extra-index-url",
                        "https://abetlen.github.io/llama-cpp-python/whl/cu122"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("CUDA wheel failed, CPU wheel:", r.stderr[-200:])
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python[server]"], check=True)

# 4b. Download the model (~2.1 GB, once per session)
if not os.path.exists(MODEL_FILE):
    print("downloading", os.path.basename(MODEL_FILE), "(~9 GB for 14B / 2.1 GB for 3B)...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_FILE)
print("model:", os.path.getsize(MODEL_FILE), "bytes")

# 4c. Launch llama-server on :58099 (background) — the port the MVM expects
def server_up():
    try:
        with urllib.request.urlopen("http://127.0.0.1:58099/v1/models", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

if not server_up():
    ngl = "-1" if GPU else "0"   # full GPU offload if a GPU runtime is attached
    cmd = [sys.executable, "-m", "llama_cpp.server", "--model", MODEL_FILE,
           "--host", "127.0.0.1", "--port", "58099", "--n_gpu_layers", ngl,
           "--n_ctx", "8192", "--chat_format", "chatml"]
    log = open("/content/llama_server.log", "w")
    subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    for i in range(120):
        if server_up():
            break
        time.sleep(2)
    else:
        print("⚠️ server not up yet — check /content/llama_server.log")
print("local LLM ready" if server_up() else "LLM NOT READY", "→ http://127.0.0.1:58099")


In [ ]:
# 5. Load the three agent routines (real MIT sources) + seed their personalities
astrid_src = open(ASTRID + "/src/astrid.m", encoding="utf-8").read()
iris_src   = open(IRIS   + "/src/iris.m",   encoding="utf-8").read()
elena_src  = open(ELENA  + "/src/elena.m",  encoding="utf-8").read()
ROUTINES = {"ASTRID": astrid_src, "IRIS": iris_src, "ELENA": elena_src}

boot = """
D INIT^ASTRID
D INIT^IRIS
D INIT^ELENA
D ASTRID^ASTRID
D IRIS^IRIS
D ELENA^ELENA
"""
out = lm.execute(boot, routines=ROUTINES, sqlite_path=DB, gas_limit=200000)
print(((out.get("state") or {}).get("output") or "").strip())


In [ ]:
# 6. ⚡ THE A·I·E CYCLE — Iris proposes · Astrid checks · Elena decides
#    Everything below runs on the real routines + the LOCAL LLM (no cloud).
TEMA = "small teams with verifiable memory make better decisions"   # ← change this and re-run!

LOCAL = ('"local", "qwen2.5"')

def m_esc(s):
    """Escape a python string as an M string literal."""
    return '"' + s.replace('"', '""') + '"'

def llm(prompt, system_key):
    """One $DEVICE llm:call with the agent's registered identity as system prompt."""
    src = f'''
S ^P = {m_esc(prompt)}
S ^S = $G(^PERSONALITY("{system_key}","identity"))
S ^R = $DEVICE("llm:call", ^P, ^S, {LOCAL})
W ^R
'''
    out = lm.execute(src, routines=ROUTINES, sqlite_path=DB, gas_limit=300000)
    return ((out.get("state") or {}).get("output") or "").strip()

def run(src):
    out = lm.execute(src, routines=ROUTINES, sqlite_path=DB, gas_limit=300000)
    return ((out.get("state") or {}).get("output") or "").strip()

print("═" * 68)
print("TOPIC:", TEMA)
print("═" * 68)

# ── 1. IRIS proposes a falsifiable hypothesis (local LLM + her identity) ──
raw = llm(
    f"Topic: {TEMA}. Propose ONE falsifiable hypothesis and its falsifier. "
    "Answer EXACTLY in this format, no extra text:\n"
    "HIPOTESIS: <one sentence>\nMECHANISM: <one sentence>\nFALSADOR: <one concrete experiment that could kill it>",
    "iris")
print("\n🌈 IRIS (creative pole) proposes:\n" + raw + "\n")

def field(txt, name, default=""):
    for line in txt.splitlines():
        if line.strip().upper().startswith(name + ":"):
            return line.split(":", 1)[1].strip() or default
    return default

hypo = field(raw, "HIPOTESIS", f"On «{TEMA}», a falsifiable effect exists")
mech = field(raw, "MECHANISM", "Unspecified mechanism (to be refined)")
fals = field(raw, "FALSADOR", "A controlled A/B test with N=20 per condition, significance p<0.05")

# register it with HER REAL ROUTINE — no falsifier, no admission
hid = run(f'W $$HYPO^IRIS({m_esc(hypo)}, {m_esc(mech)}, {m_esc(fals)}, "colab-demo")').strip()
print(f"registered in ^HYPOTHESIS → id={hid} (speculative=1, zona=open)\n")

# ── 2. ASTRID checks it against her registered evidence ──
digest = run('W $$EVIDENCE^ASTRID()')
claims = [l for l in digest.splitlines() if l.startswith("claim|")]
advice = llm(
    f"Registered evidence digest:\n{digest[:1200]}\n\n"
    f"An agent proposed this hypothesis: {hypo}\nFalsifier: {fals}\n"
    "As the evidence agent: does the registered evidence support, contradict, or say nothing about it? "
    "Answer EXACTLY:\nCONSEJO: <sostengo|refuto|no hay evidencia> — <one-sentence reason citing a claim if possible>",
    "astrid")
print("🧬 ASTRID (evidence pole) checks against " + str(len(claims)) + " claims:\n" + advice + "\n")

# ── 3. ELENA signs a decision card (her real DECIDE lint enforces the contract) ──
verdict = field(advice, "CONSEJO", "no hay evidencia")
rationale = f"Iris: {hypo} | Astrid: {verdict}"
card = run(
    f'W $$DECIDE^ELENA('
    f'{m_esc("Test the hypothesis raised by Iris on a controlled pilot")}, '
    f'{m_esc(rationale)}, '
    f'{m_esc("1) Run the pilot. 2) Archive the hypothesis without action.")}, '
    f'{m_esc("If wrong: the pilot budget is spent and the hypothesis returns to the backlog")}, '
    f'{m_esc("Pilot completed with recorded counters and a verdict on the hypothesis")}, '
    f'{m_esc("Hermes (operator)")}, '
    f'{m_esc("2026-09-17")}, '
    f'{m_esc("Reversible: the pilot creates no commitments beyond itself")}, '
    f'{m_esc("0.6")}, '
    f'{m_esc("If Astrid registers contradicting evidence before launch")})').strip()
print("🧭 ELENA (decision pole) signs:\n" + card + "\n")

# ── 4. The whole society status, straight from the PDB ──
print("═" * 68)
print(run("D IRIS^IRIS\nW \"\"\nD ELENA^ELENA\nW \"\"\nD LIST^IRIS"))
print("═" * 68)
print("Everything above is recorded in the PDB: hypothesis →", hid)


In [ ]:
# 7. Notary contract: stability + content address
import hashlib, hmac
out1 = lm.execute("W $$EVIDENCE^ASTRID()", routines=ROUTINES, sqlite_path=DB, gas_limit=80000)
d1 = ((out1.get("state") or {}).get("output") or "")
out2 = lm.execute("W $$EVIDENCE^ASTRID()", routines=ROUTINES, sqlite_path=DB, gas_limit=80000)
d2 = ((out2.get("state") or {}).get("output") or "")
cid1, cid2 = hashlib.sha256(d1.encode()).hexdigest(), hashlib.sha256(d2.encode()).hexdigest()
print("stable digest:", d1 == d2)
print("cid:", cid1[:24] + "…")
assert d1 == d2 and cid1 == cid2

demo_key = "colab-demo-key"
ts = str(int(time.time()))
sig = hmac.new(demo_key.encode(), (ts + cid1 + demo_key).encode(), hashlib.sha256).hexdigest()
lines = [ln for ln in d1.split("\n") if ln]
sets = ['S ^EVIDENCE("' + cid1 + '")="' + sig + "|" + ts + '|EVIDENCE^ASTRID"']
for i, ln in enumerate(lines, start=1):
    sets.append('S ^EVIDENCE("' + cid1 + '","digest",' + str(i) + ')="' + ln.replace('"', '""') + '"')
lm.execute("\n".join(sets), routines={}, sqlite_path=DB, gas_limit=100000)
chk = lm.execute('W $D(^EVIDENCE("' + cid1 + '"))', routines={}, sqlite_path=DB, gas_limit=5000)
print("ledger ^EVIDENCE(cid) present:", "11" in ((chk.get("state") or {}).get("output") or ""))
print("anchored:", len(lines), "digest lines under ^EVIDENCE(cid,\"digest\",n)")


In [ ]:
# 8. Bonus — semantic search over the digest claims (local embeddings, zero cost)
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "fastembed", "onnxruntime", "numpy"], check=True, timeout=300)
except Exception as e:
    print("pip install failed:", e)
try:
    t0 = time.time()
    from fastembed import TextEmbedding
    model_e = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")
    docs = claims + ["RULE: answer only from registered data"]
    vecs = list(model_e.embed(docs))
    print("embedded", len(vecs), "lines, dim:", len(vecs[0]), f"({time.time()-t0:.0f}s)")
    import numpy as np
    Q = "which MCP workers are registered?"
    qv = list(model_e.embed([Q]))[0]
    scores = [float(np.dot(qv, v) / (np.linalg.norm(qv) * np.linalg.norm(v))) for v in vecs]
    top = sorted(zip(scores, docs), key=lambda x: -x[0])[:3]
    print("query:", Q)
    for s, d in top:
        print(f"  {s:.3f}  {d[:90]}")
except Exception as e:
    print("semantic step skipped (optional):", type(e).__name__, str(e)[:120])


In [ ]:
# 9. 🎛 ASK THE SYSTEM — conversational chat: pick a sibling (or the council)
#    Real conversation: the chat history is passed to the agent, answered with
#    her registered identity + the local LLM. Every exchange is recorded in the PDB.
import subprocess, sys
try:
    import gradio as gr
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
    import gradio as gr

_CHAT_N = [0]
_EMOJI = {"iris": "🌈", "astrid": "🧬", "elena": "🧭"}
_MODE = {"🌈 Iris": "iris", "🧬 Astrid": "astrid", "🧭 Elena": "elena", "⚖️ Council (all three)": None}

_TASKS = {
    "iris":   " Eres Iris, la creativa, y estas charlando con un visitante del notebook. Responde en el idioma del visitante, con tu caracter (energia, 'y si...', posibilidades), 3-5 frases utiles sin relleno. Manten el hilo de la conversacion.",
    "astrid": " Eres Astrid, la de la evidencia, y estas charlando con un visitante. Responde en el idioma del visitante, 3-5 frases. Si afirmas algo, cita de donde viene; si no hay evidencia registrada, dilo claro. No inventes. Manten el hilo.",
    "elena":  " Eres Elena, la Decidora, y estas charlando con un visitante. Responde en el idioma del visitante, 3-5 frases, directa: si procede, decide o recomienda con criterio (y fecha si aplica); si no es tu jurisdiccion, dilo. Manten el hilo.",
}

def _hist_text(history, turns=4):
    """Flatten the chat history into one line of context (M-string safe)."""
    parts = []
    for turn in (history or [])[-turns:]:
        if isinstance(turn, (list, tuple)) and len(turn) == 2:
            u, b = turn
        elif isinstance(turn, dict):
            u, b = turn.get("content", ""), ""
        else:
            continue
        parts.append("Visitante: " + str(u)[:280].replace("\n", " "))
        if b:
            parts.append("Sistema: " + str(b)[:280].replace("\n", " "))
    return " | ".join(parts)[:1400]

def _voice(agent, question, hist_text, extra=""):
    """One agent answers, using her registered identity + the local LLM."""
    src = f'''
S ^Q = {m_esc(question)}
S ^E = {m_esc(extra)}
S ^H = {m_esc(hist_text)}
S ^T = {m_esc(_TASKS[agent])}
S ^S = $G(^PERSONALITY("{agent}","identity"))
S ^P = ^T_" | Conversacion: "_^H_" | Consulta: "_^Q
S ^R = $DEVICE("llm:call", ^P, ^S, "local", "qwen2.5")
W ^R
'''
    out = lm.execute(src, routines=ROUTINES, sqlite_path=DB, gas_limit=300000)
    return ((out.get("state") or {}).get("output") or "").strip()

def ask(question, history, quien):
    q = (question or "").strip()
    if not q:
        return "…"
    key = _MODE.get(quien, "iris")
    keys = [key] if key else ["iris", "astrid", "elena"]
    h = _hist_text(history) or "(inicio de la conversacion)"
    digest = run('W $$EVIDENCE^ASTRID()')
    ev = "Evidencia registrada: " + " | ".join(digest.splitlines()[:12]) + " | "
    replies = {}
    for k in keys:
        replies[k] = _voice(k, q, h, ev if k == "astrid" else "")
    _CHAT_N[0] += 1
    i = _CHAT_N[0]
    rec = ('S ^DEMO("chat",' + str(i) + ',"q")=' + m_esc(q) +
           '\nS ^DEMO("chat",' + str(i) + ',"quien")=' + m_esc(quien))
    for k, r in replies.items():
        rec += '\nS ^DEMO("chat",' + str(i) + ',"' + k + '")=' + m_esc(r)
    run(rec)
    if len(keys) == 1:
        body = replies[keys[0]]
    else:
        body = "\n\n".join("### " + _EMOJI[k] + " " + k.capitalize() + "\n" + replies[k] for k in keys)
    return body + "\n\n<sub>recorded in ^DEMO(\"chat\"," + str(i) + ")</sub>"

gr.ChatInterface(
    fn=ask,
    additional_inputs=[gr.Dropdown(choices=["🌈 Iris", "🧬 Astrid", "🧭 Elena", "⚖️ Council (all three)"],
                                   value="🌈 Iris", label="Talk to…")],
    title="◆ Ask the system",
    description="Pick a sibling and hold a real conversation — Iris imagines, Astrid cites the record, Elena decides. Local LLM, no cloud, no keys.",
    examples=["Hola, ¿quiénes sois y qué hace este sistema?",
              "¿Por qué no os inventáis datos cuando no hay evidencia?"],
).launch()


In [ ]:
# 10. 🚀 FULL ARSENAL — MVM ↔ MCP (native) + thinking tools + web + agents deciding with it
#     (1) launch a local LUMEN MCP server (stdlib) — system tools + thinking kit, all on the real PDB
#     (2) register it in the PDB neighbourhood and discover it from M
#     (3) the MVM interrogates the system via $DEVICE("mcp:call") — Rust JSON-RPC, no bridge
#     (4) the MVM browses the web via $DEVICE("http:get")
#     (5) Elena decides with that data in hand — decision card, on the record
#     (6) the agents' thinking kit: thought chains, patterns, decision log — via MCP, into the PDB
import subprocess, sys, time, os, json, urllib.request

_t0 = time.time()
_ACTA = []
def _paso(n, txt):
    _ACTA.append((n, round(time.time() - _t0, 1)))
    print(f"▸ {n} {txt}")

# ── 1/6 · Local LUMEN MCP server (stdlib) — system tools + thinking kit on this notebook's PDB ──
MCP_PORT = 8788
MCP_FILE = "/content/mcp_consejo.py"
MCP_SRC = r'''
import json, sqlite3, sys
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
DB = "/content/lumen_demo.db"
sys.path.insert(0, "/content/lumen-protocol/implementations/mcp-servers/pdb")
import lumen_mlight as lm

def q(sql, args=()):
    con = sqlite3.connect(DB)
    try:
        cur = con.execute(sql, args)
        cols = [c[0] for c in cur.description]
        return [dict(zip(cols, row)) for row in cur.fetchall()]
    finally:
        con.close()

def mexec(code):
    r = lm.execute(code, routines={}, sqlite_path=DB, gas_limit=80000)
    return ((r.get("state") or {}).get("output") or "").strip()

def m_esc(s):
    return '"' + str(s).replace('"', '""') + '"'

# — system tools (read) —
def sistema_estado():
    def c(ns): return q("SELECT COUNT(*) n FROM _globals WHERE ns=?", (ns,))[0]["n"]
    return {"hipotesis": c("HYPOTHESIS"), "decisiones": c("DECISION"), "pensamientos": c("THINKING"),
            "patrones": c("PATTERNS"), "personalidades": c("PERSONALITY")}
def sistema_inventario():
    return q("SELECT ns, COUNT(*) n FROM _globals GROUP BY ns ORDER BY n DESC LIMIT 10")
def sistema_acta():
    return q("SELECT value FROM _globals WHERE ns='HYPOTHESIS' LIMIT 3")

# — thinking kit (write, straight into the shared PDB) —
def thought_chain(a):
    txt, agent = str(a.get("text", "")), str(a.get("agent", "?"))
    n = mexec('S ^T=+$G(^THINKING("demo")) S ^THINKING("demo")=^T+1 S ^THINKING("demo",^T+1,"agent")=' + m_esc(agent) +
              ' S ^THINKING("demo",^T+1,"thought")=' + m_esc(txt) + ' W ^T+1')
    return {"thought_n": n, "agent": agent}
def pattern_record(a):
    name, desc = str(a.get("name", "")), str(a.get("description", ""))
    mexec('S ^PATTERNS(' + m_esc(name) + ')=' + m_esc(desc))
    return {"recorded": name}
def decision_log(a):
    did, dec = str(a.get("id", "")), str(a.get("decision", ""))
    mexec('S ^DECISION_LOG(' + m_esc(did) + ')=' + m_esc(dec))
    return {"logged": did}
def wiki_write(a):
    title, content = str(a.get("title", "")), str(a.get("content", ""))
    mexec('S ^WIKI(' + m_esc(title) + ')=' + m_esc(content))
    return {"page": title}
def ping(): return "pong"

TOOLS = [
    {"name": "sistema_estado", "description": "Estado del sistema (conteos de la PDB)", "inputSchema": {"type": "object"}},
    {"name": "sistema_inventario", "description": "Inventario de namespaces", "inputSchema": {"type": "object"}},
    {"name": "sistema_acta", "description": "Ultimas hipotesis registradas", "inputSchema": {"type": "object"}},
    {"name": "thought_chain", "description": "Añade un pensamiento a la cadena (thinking kit)", "inputSchema": {"type": "object", "properties": {"text": {"type": "string"}, "agent": {"type": "string"}}}},
    {"name": "pattern_record", "description": "Registra un patron reutilizable", "inputSchema": {"type": "object", "properties": {"name": {"type": "string"}, "description": {"type": "string"}}}},
    {"name": "decision_log", "description": "Registra una decision en el log", "inputSchema": {"type": "object", "properties": {"id": {"type": "string"}, "decision": {"type": "string"}}}},
    {"name": "wiki_write", "description": "Escribe una pagina de wiki", "inputSchema": {"type": "object", "properties": {"title": {"type": "string"}, "content": {"type": "string"}}}},
    {"name": "ping", "description": "pong", "inputSchema": {"type": "object"}},
]
_HANDLERS = {"sistema_estado": lambda a: sistema_estado(), "sistema_inventario": lambda a: sistema_inventario(),
             "sistema_acta": lambda a: sistema_acta(), "thought_chain": thought_chain, "pattern_record": pattern_record,
             "decision_log": decision_log, "wiki_write": wiki_write, "ping": lambda a: ping()}

class H(BaseHTTPRequestHandler):
    def do_POST(self):
        n = int(self.headers.get("Content-Length", 0))
        try: req = json.loads(self.rfile.read(n).decode())
        except Exception: req = {}
        rid, method, params = req.get("id", 1), req.get("method", ""), (req.get("params") or {})
        if method == "initialize":
            result = {"protocolVersion": "2024-11-05", "capabilities": {"tools": {}},
                      "serverInfo": {"name": "lumen-consejo", "version": "1.1"}}
        elif method == "tools/list":
            result = {"tools": TOOLS}
        elif method == "tools/call":
            name = params.get("name", "")
            fn = _HANDLERS.get(name)
            try:
                out = fn(params.get("arguments") or {}) if fn else {"error": "tool desconocida " + name}
            except Exception as e:
                out = {"error": str(e)[:200]}
            result = {"content": [{"type": "text", "text": json.dumps(out, ensure_ascii=False)}]}
        elif method == "ping":
            result = {}
        else:
            result = {"content": [{"type": "text", "text": "metodo desconocido"}]}
        body = json.dumps({"jsonrpc": "2.0", "id": rid, "result": result}).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)
    def log_message(self, *a): pass
print("lumen-consejo MCP on :8788")
ThreadingHTTPServer(("127.0.0.1", 8788), H).serve_forever()
'''
with open(MCP_FILE, "w", encoding="utf-8") as f:
    f.write(MCP_SRC)

def _mcp_up():
    try:
        body = json.dumps({"jsonrpc": "2.0", "id": 1, "method": "ping"}).encode()
        req = urllib.request.Request(f"http://127.0.0.1:{MCP_PORT}/mcp", data=body,
                                     headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

if not _mcp_up():
    log = open("/content/mcp_consejo.log", "w")
    subprocess.Popen([sys.executable, MCP_FILE], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(30):
        if _mcp_up():
            break
        time.sleep(0.5)
_paso("1/6", f"MCP server 'consejo' live on 127.0.0.1:{MCP_PORT} " + ("✓" if _mcp_up() else "✗ (see /content/mcp_consejo.log)"))

# ── 2/6 · Register it in the PDB neighbourhood and discover it from M ──
def run(src):
    out = lm.execute(src, routines=ROUTINES, sqlite_path=DB, gas_limit=300000)
    return ((out.get("state") or {}).get("output") or "").strip()

run('S ^SYS("MCP","consejo","url")="http://127.0.0.1:8788/mcp"\nS ^SYS("MCP","consejo","type")="http"')
lst = run('W $DEVICE("mcp:list")')
tools = run('W $DEVICE("mcp:tools","consejo")')
_paso("2/6", 'registered in ^SYS("MCP","consejo") — native discovery from M:')
print("     mcp:list  →", lst[:200])
print("     mcp:tools →", tools[:240])

# ── 3/6 · The MVM interrogates the system through MCP (native call) ──
estado_raw = run('W $DEVICE("mcp:call","consejo","sistema_estado","{}")')
_paso("3/6", "the MVM interrogates the system through MCP:")
print("     sistema_estado →", estado_raw[:260])

# ── 4/6 · The MVM browses the web (native HTTP device) ──
gh_raw = run('W $DEVICE("http:get","https://api.github.com/repos/GonzaloMonzonC/lumen-protocol")')
stars = "?"
try:
    gh = json.loads(gh_raw)
    stars = json.loads(gh["body"]).get("stargazers_count", "?")
except Exception:
    pass
_paso("4/6", "the MVM fetches the web (HTTP device):")
print("     api.github.com · lumen-protocol → ⭐", stars)

# ── 5/6 · Elena decides with that data — decision card, on the record ──
src5 = f'''
S ^EST = {m_esc(estado_raw[:400])}
S ^GH = {m_esc("stars=" + str(stars))}
S ^P = "Datos del sistema (via MCP): "_^EST_" | Popularidad (via HTTP): "_^GH_" | Pregunta: como Decidora, el sistema esta listo para mostrarse al mundo? Responde en 2 frases."
S ^S = $G(^PERSONALITY("elena","identity"))
S ^V = $DEVICE("llm:call", ^P, ^S, "local", "qwen2.5")
W ^V
'''
veredicto = run(src5)
print("     🧭 Elena (con los datos del MCP y la web): " + veredicto[:320])

card = run('W $$DECIDE^ELENA("Mostrar el sistema al mundo (demo Colab)",'
           + m_esc("Estado via MCP: " + estado_raw[:200] + " | GitHub stars: " + str(stars) + " | Elena: " + veredicto[:200]) + ','
           + m_esc("1) Publicar la demo ya. 2) Pulir mas antes.") + ','
           + m_esc("Si salen criticas, se itera en publico — es reversible") + ','
           + m_esc("Demo publica viva: ciclo A-I-E + arsenal MCP + LLM local") + ','
           + m_esc("Gonzalo") + ','
           + m_esc("2026-09-24") + ','
           + m_esc("Reversible: la demo siempre se puede retirar") + ','
           + m_esc("0.7") + ','
           + m_esc("Si la comunidad detecta fallos de honestidad en los agentes") + ')')
_paso("5/6", "Elena signs, data in hand: " + card.strip()[:100])

# ── 6/6 · The thinking kit via MCP — the agents' own reasoning, into the shared PDB ──
t1 = run('W $DEVICE("mcp:call","consejo","thought_chain",' + m_esc(json.dumps({"text": f"Evalúo mostrar el demo: estado del sistema + {stars} stars. Suficiente para publicar.", "agent": "elena"})) + ')')
t2 = run('W $DEVICE("mcp:call","consejo","thought_chain",' + m_esc(json.dumps({"text": "Cadena de pensamiento en la PDB: si mañana hay +evidencia, se revisa (revisit trigger está en la card).", "agent": "astrid"})) + ')')
p1 = run('W $DEVICE("mcp:call","consejo","pattern_record",' + m_esc(json.dumps({"name": "arsenal-mcp-demo", "description": "MVM + MCP nativo + thinking kit + LLM local: la artillería del cuaderno"})) + ')')
d1 = run('W $DEVICE("mcp:call","consejo","decision_log",' + m_esc(json.dumps({"id": card.strip()[:40], "decision": "Publicar demo v2 con arsenal completo"})) + ')')
_paso("6/6", "thinking kit via MCP (thoughts + pattern + decision log):")
print("     thought_chain →", t1[:120])
print("     thought_chain →", t2[:120])
print("     pattern_record →", p1[:100])
print("     decision_log →", d1[:100])
estado2 = run('W $DEVICE("mcp:call","consejo","sistema_estado","{}")')
print("     sistema_estado (después) →", estado2[:240])

print()
print("═" * 68)
print("ACTA TÉCNICA — medido en este run (t+ segundos desde el inicio de la celda):")
for n, t in _ACTA:
    print(f"   {n:6s} t+{t:5.1f}s")
print("═" * 68)
print("MCP nativo + HTTP nativo + thinking kit + agentes decidiendo. Todo en la PDB, todo con acta.")


In [ ]:
# 11. ⚡ THE COUNCIL IN PARALLEL — $FORK ×3: three voices, one local LLM, at once
#     The MVM speaks M natively: fork the three siblings into fibers, await them,
#     and compare against the sequential round.
import time

PREGUNTA = "¿Qué mejora primero en el sistema: más evidencia, más ideas o más decisiones firmadas? Responde en 2 frases, en tu voz."

def _voice_prompt(agent, task):
    return f'S ^P{n[agent]} = {m_esc(task + " | Pregunta del visitante: " + PREGUNTA)}'

n = {"iris": 1, "astrid": 2, "elena": 3}
src_fib = f'''
S ^SI = $G(^PERSONALITY("iris","identity"))
S ^SA = $G(^PERSONALITY("astrid","identity"))
S ^SE = $G(^PERSONALITY("elena","identity"))
S ^P1 = {m_esc("Eres Iris, la creativa. " + PREGUNTA + " Responde 2 frases con tu 'y si...'.")}
S ^P2 = {m_esc("Eres Astrid, la de la evidencia. " + PREGUNTA + " Responde 2 frases; si no hay evidencia, dilo.")}
S ^P3 = {m_esc("Eres Elena, la Decidora. " + PREGUNTA + " Responde 2 frases con criterio.")}
S ^F1 = $FORK(^P1, ^SI, "local", "qwen2.5")
S ^F2 = $FORK(^P2, ^SA, "local", "qwen2.5")
S ^F3 = $FORK(^P3, ^SE, "local", "qwen2.5")
S ^R1 = $AWAIT(^F1)
S ^R2 = $AWAIT(^F2)
S ^R3 = $AWAIT(^F3)
W ^R1,!,!,^R2,!,!,^R3,!
'''

t0 = time.time()
out_fib = lm.execute(src_fib, routines=ROUTINES, sqlite_path=DB, gas_limit=400000)
t_fib = time.time() - t0
resp_fib = ((out_fib.get("state") or {}).get("output") or "").strip()

t0 = time.time()
s1 = run(f'S ^X=$DEVICE("llm:call", ^P1, ^SI, "local", "qwen2.5") W ^X')
t_seq1 = time.time() - t0

print("═" * 68)
print("⚡ CONSEJO EN FIBRAS — $FORK ×3 + $AWAIT (mismo MVM, mismo LLM local)")
print("═" * 68)
print(resp_fib[:900])
print()
print(f"⏱  una voz secuencial: {t_seq1:.1f}s  ·  las tres en fibras: {t_fib:.1f}s"
      f"  →  speedup ≈ {max(t_seq1, 0.01) * 3 / max(t_fib, 0.01):.1f}× vs secuencial")
print("El modelo de concurrencia del MVM: código M que lanza trabajos y los espera. Sin threads en el host.")


In [ ]:
# 12. ⬡ THE BROWSER CONSOLE — your MVM, in your browser, wired to THIS notebook's PDB
#     - downloads the console + WASM package from the repo's Release (self-contained)
#     - launches a vm_api instance bound to this notebook's PDB (DDP REST)
#     - a tiny same-origin proxy serves the console and forwards /ddp/* to it
#     - the console renders inline (Colab port-forwarding) — write M live:
#         S ^HOLA="mundo"   ·   ZW ^HYPOTHESIS   ·   D ^%GL   ·   ^%GD ^NS
import subprocess, sys, time, os, json, urllib.request, zipfile, socket

CONSOLE_DIR = "/content/console"
ZIP_URL = "https://github.com/GonzaloMonzonC/lumen-protocol/releases/download/mlight-v0.1.0/mvm-console-pkg.zip"
ZIP_FILE = "/content/mvm-console-pkg.zip"
if not os.path.exists(CONSOLE_DIR + "/m_light_console.html"):
    os.makedirs(CONSOLE_DIR, exist_ok=True)
    urllib.request.urlretrieve(ZIP_URL, ZIP_FILE)
    with zipfile.ZipFile(ZIP_FILE) as z:
        z.extractall(CONSOLE_DIR)
print("console files:", sorted(os.listdir(CONSOLE_DIR)))

def port_up(p):
    s = socket.socket(); s.settimeout(1)
    try:
        s.connect(("127.0.0.1", p)); return True
    except Exception:
        return False
    finally:
        s.close()

# ── vm_api :8090 — the same engine, bound to the notebook's PDB ──
VM_API = "/content/lumen-protocol/implementations/python/pdb-sync/vm_api.py"
env = dict(os.environ)
env.pop("DDP_HMAC_KEY", None)                   # dev mode: pull/push sin gate HMAC (lección conocida)
env.pop("PDB_MACAROON_REQUIRED", None)
env["PDB_PATH"] = DB
env["LUMEN_MLIGHT_LIB"] = str(lm._lib_path())   # the .so this notebook already downloaded
env["DASHBOARD_TOKEN"] = "colab-demo-token"     # the proxy will present it to /ddp/*
if not port_up(8090):
    log = open("/content/vm_api_console.log", "w")
    subprocess.Popen([sys.executable, VM_API, "8090"], cwd=os.path.dirname(VM_API), env=env,
                     stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        if port_up(8090):
            break
        time.sleep(0.5)
print("vm_api :8090", "✓" if port_up(8090) else "✗ (see /content/vm_api_console.log)")

# ── mini proxy :8092 — serves the console, forwards /ddp/* to the vm_api (same origin) ──
PROXY = "/content/console_proxy.py"
PROXY_SRC = '''
import http.server, urllib.request, os
CD = "/content/console"
class H(http.server.SimpleHTTPRequestHandler):
    def __init__(self, *a, **k):
        super().__init__(*a, directory=CD, **k)
    def _proxy(self):
        token = "colab-demo-token"
        sep = "&" if "?" in self.path else "?"
        url = "http://127.0.0.1:8090" + self.path + sep + "t=" + token
        ln = int(self.headers.get("Content-Length", 0))
        body = self.rfile.read(ln) if ln else None
        req = urllib.request.Request(url, data=body, method=self.command,
                                     headers={"Content-Type": self.headers.get("Content-Type", "application/json")})
        try:
            with urllib.request.urlopen(req, timeout=120) as r:
                data = r.read()
                self.send_response(r.status)
                self.send_header("Content-Type", r.headers.get("Content-Type", "application/json"))
                self.send_header("Content-Length", str(len(data)))
                self.end_headers()
                self.wfile.write(data)
        except Exception as e:
            msg = str(e).encode()
            self.send_response(502)
            self.send_header("Content-Length", str(len(msg)))
            self.end_headers()
            self.wfile.write(msg)
    def do_GET(self):
        if self.path.startswith("/ddp"):
            self._proxy()
        else:
            super().do_GET()
    def do_POST(self):
        if self.path.startswith("/ddp"):
            self._proxy()
        else:
            self.send_error(405)
    def log_message(self, *a):
        pass
print("console proxy :8092")
http.server.ThreadingHTTPServer(("127.0.0.1", 8092), H).serve_forever()
'''
with open(PROXY, "w", encoding="utf-8") as f:
    f.write(PROXY_SRC)
if not port_up(8092):
    log2 = open("/content/console_proxy.log", "w")
    subprocess.Popen([sys.executable, PROXY], stdout=log2, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(20):
        if port_up(8092):
            break
        time.sleep(0.5)
print("proxy :8092", "✓" if port_up(8092) else "✗ (see /content/console_proxy.log)")

try:
    from google.colab import output as _co
    _co.serve_kernel_port_as_iframe(8092, height=480)
    print("⬡ Consola embebida. Escribe M en vivo, p.ej.:")
    print('   S ^HOLA="mundo"   ·   ZW ^HYPOTHESIS   ·   ZW ^THINKING   ·   D ^%GL')
except Exception as e:
    print("(sin Colab: abre http://127.0.0.1:8092 en el navegador)", e)


## What you just ran

- **MVM**: a Rust virtual machine executed real M code (seeding, hypothesis registration, decision lint, ledger writes) — deterministic, gated by gas.
- **PDB**: hierarchical, transactional memory (SQLite) shared by the three agents.
- **A·I·E society** — three real MIT agents with contracts:
  - 🌈 **Iris** turns topics into *falsifiable* hypotheses (no falsifier, no admission).
  - 🧬 **Astrid** answers only from registered evidence (`claim|kind|source|value|d` — no source, no claim).
  - 🧭 **Elena** converts "truth enough + options" into a signed **decision card**: criterion, owner, review date — without them, it is not a decision.
- **The LLM lived here**: Qwen2.5-14B (3B fallback on CPU) served by llama.cpp on `127.0.0.1:58099`, reached through the MVM's native `$DEVICE("llm:call", …, "local", …)`. No keys, no cloud, no bill.
- **Notary seed**: same state → same digest → same `cid`.
- **Full arsenal**: the MVM called a local **MCP server** (native JSON-RPC from Rust — `mcp:list`, `mcp:tools`, `mcp:call`), fetched GitHub over its **HTTP device**, and Elena signed a decision card informed by that live data.
- **Ask the system**: the interactive cell lets you interrogate the society yourself — Iris imagines, Astrid checks the record, Elena decides. Every Q&A lands in `^DEMO("chat",n)`.

Next steps: fork [Astrid's template](https://github.com/GonzaloMonzonC/astrid/tree/main/template) to build your own agent, run the full suites ([astrid](https://github.com/GonzaloMonzonC/astrid) · [iris](https://github.com/GonzaloMonzonC/iris) · [elena](https://github.com/GonzaloMonzonC/elena)), or read the [protocol docs](https://github.com/GonzaloMonzonC/lumen-protocol/tree/main/docs).

*Demo data is fictional — the notebook is fully self-contained and MIT-clean.*
